In [1]:
from google.colab import files
files.upload()

Saving entities_alignment.py to entities_alignment.py


{'entities_alignment.py': b'# -*- coding: utf-8 -*-\n"""\n\xd8\xae\xd8\xb7 \xd9\x85\xd8\xb9\xd8\xa7\xd9\x84\xd8\xac\xd8\xa9: \xd8\xaa\xd9\x86\xd8\xb8\xd9\x8a\xd9\x81 \xd9\x86\xd8\xb5 \xd8\xb9\xd8\xb1\xd8\xa8\xd9\x8a -> \xd8\xaa\xd9\x82\xd8\xb7\xd9\x8a\xd8\xb9 \xd8\xa5\xd9\x84\xd9\x89 \xd8\xac\xd9\x85\xd9\x84 -> \xd8\xaa\xd8\xb1\xd8\xac\xd9\x85\xd8\xa9 (Helsinki) -> \xd8\xa7\xd8\xb3\xd8\xaa\xd8\xae\xd8\xb1\xd8\xa7\xd8\xac \xd9\x83\xd9\x8a\xd8\xa7\xd9\x86\xd8\xa7\xd8\xaa \xd8\xb9\xd8\xb1\xd8\xa8\xd9\x8a\xd8\xa9/\xd8\xa5\xd9\x86\xd8\xac\xd9\x84\xd9\x8a\xd8\xb2\xd9\x8a\xd8\xa9\n-> \xd9\x85\xd9\x88\xd8\xa7\xd8\xa1\xd9\x85\xd8\xaa\xd9\x87\xd8\xa7 -> \xd8\xb5\xd9\x81\xd8\xad\xd8\xa9 HTML \xd9\x8a\xd9\x88\xd9\x85\xd8\xb6 \xd9\x81\xd9\x8a\xd9\x87\xd8\xa7 \xd8\xa7\xd9\x84\xd9\x83\xd9\x8a\xd8\xa7\xd9\x86 \xd9\x88\xd9\x86\xd8\xb8\xd9\x8a\xd8\xb1\xd9\x87 \xd8\xb9\xd9\x86\xd8\xaf \xd8\xaa\xd9\x85\xd8\xb1\xd9\x8a\xd8\xb1 \xd8\xa7\xd9\x84\xd9\x85\xd8\xa7\xd9\x88\xd8\xb3.\n\n\xd8\xa7\xd9\x84\xd8\xaa\xd

In [2]:
!pip install -q transformers sentencepiece spacy datasets
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 79.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import re
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus-100", "ar-en", split="train", streaming=True)
lines, words = [], 0
for ex in ds:
    ar = ex["translation"]["ar"].strip()
    if len(ar.split()) < 6:
        continue
    if not re.search(r"[.!؟?]$", ar):
        ar += "."
    lines.append(ar)
    words += len(ar.split())
    if words >= 1200:
        break

open("input.txt", "w", encoding="utf-8").write("\n".join(lines))
print("عدد الجمل:", len(lines), "| عدد الكلمات:", words)

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

عدد الجمل: 78 | عدد الكلمات: 1208


In [11]:
new_block = '''    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    tok = AutoTokenizer.from_pretrained(MT_MODEL)
    mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL)
    def translate_batch(xs):
        out = []
        for i in range(0, len(xs), 16):
            b = tok(xs[i:i+16], return_tensors="pt", padding=True, truncation=True, max_length=256)
            with torch.no_grad():
                g = mt.generate(**b, max_new_tokens=256, num_beams=4)
            out += tok.batch_decode(g, skip_special_tokens=True)
        return out'''

lines = open("entities_alignment.py", encoding="utf-8").read().splitlines()

i = next((k for k, l in enumerate(lines) if 'pipeline("translation"' in l), None)
j = next((k for k, l in enumerate(lines) if "def translate_batch" in l), None)

if i is None or j is None:
    print("لم أجد السطرين. جرّبي تنفيذ: !grep -n translat entities_alignment.py وأرسلي لي النتيجة")
else:
    lines[i:j+1] = new_block.splitlines()
    open("entities_alignment.py", "w", encoding="utf-8").write("\n".join(lines) + "\n")
    print("تم التعديل")

لم أجد السطرين. جرّبي تنفيذ: !grep -n translat entities_alignment.py وأرسلي لي النتيجة


In [12]:
!python entities_alignment.py input.txt entities_alignment_reader.html

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100% 258/258 [00:00<00:00, 21334.96it/s]
Loading weights: 100% 199/199 [00:00<00:00, 9938.28it/s]
[transformers] BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Bo

In [13]:
from IPython.display import HTML, display
display(HTML(open("entities_alignment_reader.html", encoding="utf-8").read()))

In [14]:
files.download("entities_alignment_reader.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
%%writefile entities_alignment.py
# -*- coding: utf-8 -*-
import re, sys, html, torch
from difflib import SequenceMatcher
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import spacy

MT_MODEL = "Helsinki-NLP/opus-mt-ar-en"
AR_NER_MODEL = "CAMeL-Lab/bert-base-arabic-camelbert-msa-ner"
THRESHOLD = 0.35

def clean(text):
    text = re.sub(r"[\u064B-\u0652\u0670]", "", text)
    text = text.replace("\u0640", "")
    text = re.sub(r"[\u200b-\u200f\u202a-\u202e\ufeff]", "", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[ \t\r\n]+", " ", text)
    text = re.sub(r"\s+([.،؛:!؟?])", r"\1", text)
    return text.strip()

def split_sentences(text):
    parts = re.split(r"(?<=[.!؟?])(?<!\d\.)\s+", text)
    return [p.strip() for p in parts if len(p.strip()) > 1]

MONTHS = "يناير|فبراير|مارس|أبريل|ابريل|إبريل|مايو|يونيو|يونيه|يوليو|يوليه|أغسطس|اغسطس|سبتمبر|أكتوبر|اكتوبر|نوفمبر|ديسمبر|كانون الثاني|كانون الأول|شباط|آذار|نيسان|أيار|حزيران|تموز|آب|أيلول|تشرين الأول|تشرين الثاني|محرم|صفر|ربيع الأول|ربيع الآخر|جمادى الأولى|جمادى الآخرة|رجب|شعبان|رمضان|شوال|ذو القعدة|ذو الحجة"
DAYS = "الأحد|الاثنين|الإثنين|الثلاثاء|الأربعاء|الخميس|الجمعة|السبت"
ORD = "الأول|الثاني|الثالث|الرابع|الخامس|السادس|السابع|الثامن|التاسع|العاشر|الحادي عشر|الثاني عشر|الثالث عشر|الرابع عشر|الخامس عشر|السادس عشر|السابع عشر|الثامن عشر|التاسع عشر|العشرين|الحادي والعشرين"
TIME_PATTERNS = [
    rf"(?:(?:يوم\s+)?(?:{DAYS})\s+)?(?:\d{{1,2}}\s+)?(?:شهر\s+)?(?:{MONTHS})(?:\s+(?:من\s+)?(?:عام|سنة)?\s*\d{{3,4}}(?:\s*(?:م|هـ|ه))?)?",
    rf"القرن\s+(?:{ORD})(?:\s+(?:الميلادي|الهجري|قبل الميلاد))?",
    rf"(?:عام|سنة)\s+\d{{3,4}}(?:\s*(?:م|هـ|ه))?",
    rf"\b(?:{DAYS})\b",
    r"\b(?:1[0-9]{3}|20[0-9]{2})\b(?:\s*(?:م|هـ))?",
]
def time_entities(text):
    out = []
    for p in TIME_PATTERNS:
        for m in re.finditer(p, text):
            s, e = m.span()
            if e > s and not any(s < e2 and e > s2 for s2, e2, _ in out):
                out.append((s, e, "DATE"))
    return out

GROUP = {"PER": "PER", "PERSON": "PER", "LOC": "LOC", "GPE": "LOC", "FAC": "LOC",
         "ORG": "ORG", "MISC": "MISC", "NORP": "MISC", "EVENT": "MISC", "PRODUCT": "MISC",
         "WORK_OF_ART": "MISC", "LAW": "MISC", "LANGUAGE": "MISC", "DATE": "DATE", "TIME": "DATE"}

def arabic_entities(text, ner):
    ents = time_entities(text)
    for r in ner(text):
        s, e = r["start"], r["end"]
        g = GROUP.get(r["entity_group"].replace("B-", "").replace("I-", ""), "MISC")
        if not any(s < e2 and e > s2 for s2, e2, _ in ents):
            ents.append((s, e, g))
    return sorted(ents)

def english_entities(text, nlp):
    return [(x.start_char, x.end_char, GROUP[x.label_]) for x in nlp(text).ents if x.label_ in GROUP]

def norm(s): return re.sub(r"[^\w\s]", "", s.lower()).strip()
def similarity(a, b):
    a, b = norm(a), norm(b)
    if not a or not b: return 0
    ta, tb = set(a.split()), set(b.split())
    jac = len(ta & tb) / len(ta | tb)
    seq = SequenceMatcher(None, a, b).ratio()
    dig = 0.3 if set(re.findall(r"\d+", a)) & set(re.findall(r"\d+", b)) else 0
    return max(jac, seq) + dig

def align(ar_text, ar_ents, en_text, en_ents, translate_batch, cache):
    need = list({ar_text[s:e] for s, e, _ in ar_ents if ar_text[s:e] not in cache})
    if need:
        for k, v in zip(need, translate_batch(need)): cache[k] = v
    scored = []
    for i, (s, e, g) in enumerate(ar_ents):
        tr = cache[ar_text[s:e]]
        for j, (s2, e2, g2) in enumerate(en_ents):
            if g == g2 or "MISC" in (g, g2):
                sc = similarity(tr, en_text[s2:e2])
                if sc >= THRESHOLD: scored.append((sc, i, j))
    pairs, ui, uj = [], set(), set()
    for sc, i, j in sorted(scored, reverse=True):
        if i not in ui and j not in uj:
            ui.add(i); uj.add(j); pairs.append((i, j))
    return pairs

def words_html(text, ents):
    """كل كلمة في span مستقل، والكلمات التي تقع داخل كيان متواءم تأخذ data-eid"""
    out = []
    for m in re.finditer(r"\S+", text):
        eid = None
        for s, e, g, i in ents:
            if m.start() < e and m.end() > s:
                eid = i; break
        w = html.escape(m.group())
        if eid:
            out.append(f'<span class="word entity-word" data-eid="{eid}" onmouseover="hl(\'{eid}\')" onmouseout="unhl()">{w}</span>')
        else:
            out.append(f'<span class="word">{w}</span>')
    return " ".join(out)

PAGE = """<!DOCTYPE html>
<html lang="ar"><head><meta charset="utf-8"><title>الكيانات العربية ومقابلاتها الإنجليزية</title>
<style>
body { font-family: 'Segoe UI', system-ui, sans-serif; background:#f8fafc; margin:0; padding:30px; direction:rtl; }
.container { max-width:1300px; margin:0 auto; background:white; padding:40px; border-radius:16px; box-shadow:0 10px 25px rgba(0,0,0,0.05); }
h2 { color:#1e293b; text-align:center; border-bottom:2px solid #f1f5f9; padding-bottom:15px; }
.subtitle { text-align:center; color:#475569; margin-bottom:30px; }
.reader-table { width:100%; border-collapse:collapse; table-layout:fixed; }
.reader-table th { background:#f1f5f9; padding:14px; border:1px solid #e2e8f0; }
.reader-table td { padding:15px 20px; border:1px solid #e2e8f0; vertical-align:middle; width:50%; }
.reader-table tr:nth-child(even) { background:#f8fafc; }
.word { display:inline-block; margin:0 2px; padding:2px 4px; font-size:17px; border-radius:4px; transition:background-color .1s ease; }
.entity-word { cursor:pointer; border-bottom:2px dotted #94a3b8; }
.flash-active { background-color:#fde047 !important; color:#0f172a !important; font-weight:bold; box-shadow:0 0 10px #fde047; transform:scale(1.04); }
.ar-col { text-align:right; line-height:1.8; color:#0f172a; }
.en-col { text-align:left; line-height:1.8; color:#1e293b; direction:ltr; }
</style>
<script>
function hl(id){ document.querySelectorAll('[data-eid="'+id+'"]').forEach(function(e){ e.classList.add('flash-active'); }); }
function unhl(){ document.querySelectorAll('.flash-active').forEach(function(e){ e.classList.remove('flash-active'); }); }
</script></head><body><div class="container">
<h2>نظام تحديد الكيانات العربية ومقابلاتها الإنجليزية</h2>
<p class="subtitle">مرر الماوس فوق الكلمات المنقّطة لترى الوميض الأصفر على الكيان ومقابله.</p>
<table class="reader-table"><thead><tr><th>الجملة العربية</th><th>الترجمة الإنجليزية</th></tr></thead>
<tbody>%ROWS%</tbody></table></div></body></html>"""

def main(inp, outp):
    sents = split_sentences(clean(open(inp, encoding="utf-8").read()))
    tok = AutoTokenizer.from_pretrained(MT_MODEL)
    mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL)
    def translate_batch(xs):
        out = []
        for i in range(0, len(xs), 16):
            b = tok(xs[i:i+16], return_tensors="pt", padding=True, truncation=True, max_length=256)
            with torch.no_grad():
                g = mt.generate(**b, max_new_tokens=256, num_beams=4)
            out += tok.batch_decode(g, skip_special_tokens=True)
        return out
    ner = pipeline("token-classification", model=AR_NER_MODEL, aggregation_strategy="simple")
    nlp = spacy.load("en_core_web_sm")

    translations = translate_batch(sents)
    cache, rows, eid, n_ar, n_en = {}, [], 0, 0, 0
    for ar, en in zip(sents, translations):
        ar_e, en_e = arabic_entities(ar, ner), english_entities(en, nlp)
        n_ar += len(ar_e); n_en += len(en_e)
        pairs = align(ar, ar_e, en, en_e, translate_batch, cache)
        ar_w, en_w = [], []
        for i, j in pairs:
            eid += 1
            ar_w.append((ar_e[i][0], ar_e[i][1], ar_e[i][2], f"e{eid}"))
            en_w.append((en_e[j][0], en_e[j][1], ar_e[i][2], f"e{eid}"))
        rows.append(f'<tr><td class="ar-col">{words_html(ar, ar_w)}</td><td class="en-col">{words_html(en, en_w)}</td></tr>')
    open(outp, "w", encoding="utf-8").write(PAGE.replace("%ROWS%", "\n".join(rows)))
    print(f"جمل: {len(sents)} | كيانات عربية: {n_ar} | كيانات إنجليزية: {n_en} | أزواج متواءمة: {eid}")

if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])

Overwriting entities_alignment.py


In [16]:
!python entities_alignment.py input.txt entities_alignment_reader.html

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100% 258/258 [00:00<00:00, 11478.32it/s]
Loading weights: 100% 199/199 [00:00<00:00, 1495.75it/s]
[transformers] BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Bo

In [17]:
from google.colab import files
files.download("entities_alignment_reader.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>